# 06 — Warmstart Ablation: Chamfer → Power-SM3

**Hypothesis**: Chamfer loss is good at breaking symmetry early (its hard-min
ensures clear per-slot signals), but converges to suboptimal coverage.
Power-SM3 is better at full coverage but may need a good initialisation to avoid
getting stuck. Switching from Chamfer to Power-SM3 mid-training ("warm-start")
may combine both benefits.

**Experiment**: Train with ChamferLoss for `warmup_epochs` epochs then switch to
PowerSoftMinLoss τ=0.12 for the remainder. Compare warmup_epochs ∈ {0, 3, 10, 20}
(where 0 = pure Power-SM3 from scratch).


In [1]:
import sys
sys.path.insert(0, '.')
import clevr_utils as cu
import torch
import numpy as np
from influencerformer.losses import ChamferLoss, PowerSoftMinLoss
from scipy.optimize import linear_sum_assignment
print(f'Device: {cu.DEVICE}')


Device: cuda


In [2]:
WARMUP_LIST = [0, 3, 10, 20]
N_EPOCHS = 50
SNAPSHOT_EPOCHS = {0, 5, 15, 35, 49}


In [ ]:
def train_warmstart(warmup_epochs, n_epochs=50, lr=3e-4, snapshot_epochs=None, seed=42):
    '''Train with ChamferLoss for warmup_epochs then switch to PowerSoftMinLoss.

    Args:
        warmup_epochs: epochs to train with ChamferLoss (0 = pure PSM3 from scratch)
        n_epochs: total training epochs
        lr: Adam learning rate
        snapshot_epochs: set of epoch indices to save prediction snapshots
        seed: random seed for reproducibility
    Returns:
        (metrics, snapshots) in the same format as cu.train_and_monitor
    '''
    if snapshot_epochs is None:
        snapshot_epochs = {0, 5, 15, 35, n_epochs - 1}

    torch.manual_seed(seed)
    model = cu.make_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    chamfer_loss = ChamferLoss()
    psm_loss = PowerSoftMinLoss(temperature=0.12, power=3.0)

    train_loader = cu.make_dataloader('train', max_samples=5000)
    val_loader = cu.make_dataloader('val', max_samples=1000)

    metrics = {
        'epoch': [], 'train_loss': [], 'val_matched_dist': [],
        'pred_diversity': [], 'softmax_entropy': [], 'grad_diversity': [],
        'lr': [],
    }
    snapshots = []

    val_batch_fixed = next(iter(val_loader))
    inp_v, tgt_v, msk_v = [x.to(cu.DEVICE) for x in val_batch_fixed]

    for epoch in range(n_epochs):
        loss_fn = chamfer_loss if epoch < warmup_epochs else psm_loss

        model.train()
        train_losses = []
        for inp, tgt, msk in train_loader:
            inp, tgt, msk = inp.to(cu.DEVICE), tgt.to(cu.DEVICE), msk.to(cu.DEVICE)
            optimizer.zero_grad()
            preds = model(inp, mask=msk)
            D = torch.cdist(preds, tgt)
            loss = loss_fn(D, mask=msk)
            if torch.isfinite(loss):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                train_losses.append(loss.item())

        model.eval()
        val_dists, diversities, entropies = [], [], []
        with torch.no_grad():
            for inp, tgt, msk in val_loader:
                inp, tgt, msk = inp.to(cu.DEVICE), tgt.to(cu.DEVICE), msk.to(cu.DEVICE)
                preds = model(inp, mask=msk)
                val_dists.append(cu.hungarian_dist(preds, tgt, msk))
                diversities.append(cu.pred_diversity(preds, msk))
                entropies.append(cu.softmax_entropy(preds, tgt, msk, tau=0.12))

        gd = cu.grad_diversity(model, next(iter(train_loader)), loss_fn)
        current_lr = optimizer.param_groups[0]['lr']

        metrics['epoch'].append(epoch)
        metrics['train_loss'].append(float(np.mean(train_losses)))
        metrics['val_matched_dist'].append(float(np.mean(val_dists)))
        metrics['pred_diversity'].append(float(np.mean(diversities)))
        metrics['softmax_entropy'].append(float(np.mean(entropies)))
        metrics['grad_diversity'].append(gd)
        metrics['lr'].append(current_lr)

        if epoch in snapshot_epochs:
            model.eval()
            with torch.no_grad():
                preds_snap = model(inp_v, mask=msk_v).cpu().numpy()
            snapshots.append((epoch, preds_snap, tgt_v.cpu().numpy(), msk_v.cpu().numpy()))

        phase = 'chamfer' if epoch < warmup_epochs else 'psm3'
        tl = metrics['train_loss'][-1]
        vd = metrics['val_matched_dist'][-1]
        pd_ = metrics['pred_diversity'][-1]
        se = metrics['softmax_entropy'][-1]
        gd_val = metrics['grad_diversity'][-1]
        print(
            f'  [warmup={warmup_epochs}] epoch {epoch:3d} ({phase}) | '
            f'loss={tl:.4f} dist={vd:.4f} div={pd_:.3f} H={se:.3f} gd={gd_val:.3f}'
        )

    return metrics, snapshots


all_results = {}
for warmup_ep in WARMUP_LIST:
    label = f'warmup={warmup_ep}'
    print(f'=== {label} ===')
    metrics, snapshots = train_warmstart(
        warmup_epochs=warmup_ep,
        n_epochs=N_EPOCHS,
        snapshot_epochs=SNAPSHOT_EPOCHS,
    )
    all_results[label] = (metrics, snapshots)


=== warmup=0 ===


/global/homes/d/danieltm/.conda/envs/influencer/lib/python3.11/site-packages/torch/nn/modules/transformer.py:296: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /opt/conda/conda-bld/pytorch_1682343970094/work/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


  [warmup=0] epoch   0 (psm3) | loss=10.5985 dist=1.6252 div=1.487 H=0.224 gd=0.739
  [warmup=0] epoch   1 (psm3) | loss=6.9616 dist=1.5249 div=1.538 H=0.204 gd=0.591
  [warmup=0] epoch   2 (psm3) | loss=5.6338 dist=1.4429 div=1.611 H=0.211 gd=0.575
  [warmup=0] epoch   3 (psm3) | loss=4.6823 dist=1.3842 div=1.708 H=0.204 gd=0.532
  [warmup=0] epoch   4 (psm3) | loss=4.0530 dist=1.3388 div=1.774 H=0.197 gd=0.676
  [warmup=0] epoch   5 (psm3) | loss=3.6125 dist=1.3050 div=1.798 H=0.195 gd=0.659
  [warmup=0] epoch   6 (psm3) | loss=3.2976 dist=1.2851 div=1.823 H=0.190 gd=0.495
  [warmup=0] epoch   7 (psm3) | loss=3.0575 dist=1.2619 div=1.862 H=0.188 gd=0.562
  [warmup=0] epoch   8 (psm3) | loss=2.8828 dist=1.2413 div=1.869 H=0.185 gd=0.526
  [warmup=0] epoch   9 (psm3) | loss=2.7403 dist=1.2224 div=1.919 H=0.180 gd=0.565
  [warmup=0] epoch  10 (psm3) | loss=2.6335 dist=1.2117 div=1.912 H=0.180 gd=0.520
  [warmup=0] epoch  11 (psm3) | loss=2.5524 dist=1.1984 div=1.928 H=0.175 gd=0.473
  [

In [ ]:
cu.plot_monitoring(all_results, title='Warmstart Ablation: Chamfer → Power-SM3')

cu.plot_pca_snapshots(all_results['warmup=0'][1], title='warmup=0 (pure PSM3)')
cu.plot_pca_snapshots(all_results['warmup=10'][1], title='warmup=10 (Chamfer→PSM3)')


In [ ]:
cu.summary_table(all_results)
